In [1]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Computing the Conjunctive Normal Form in First Order Logic

In order to convert a formula $f$ from first order logic into a set of clauses that is satisfiable if and only if $f$ is satisfiable,
we have to perform the following steps in order:
- eliminate biconditionals,
- eliminate conditionals,
- transform the formula into *negation normal form*,
  i.e. we push the negation symbol inwards,
- rename bound variables to avoid clashes, 
- transform the formula into *prenex normal form*,
  i.e. we move the quantifieres outside,
  
- eliminate existential quantifiers by *skolemizing* the formula, and
- transform the formula into *clauses* in set notation.

When converting formulas into conjunctive normal form, we <u>assume</u> that the formulas are 
*pure*, where we define a formula $f$ as *pure* if all quantifiers appearing in $f$ bind **different** variables.  For example, the formula
$$ \bigl(\forall x: p(x)\bigr) \vee \bigl(\forall x: q(x)\bigr)$$
is **not** *pure*, because there are two different universal quantifiers that both bind the same variable $x$.  We can rewrite this formulas as a *pure* formula by *renaming* all occurrences of $x$ that are bound by the second quantifier as follows:
$$ \bigl(\forall x: p(x)\bigr) \vee \bigl(\forall y: q(y)\bigr)$$

## Auxilliary Functions

Formulas are represented as nested tuples.  In order to convert a string into a nested tuple we use the <tt>LogicParser</tt> that is found in the module <tt>FOL-Parser</tt>.  Our parser distinguishes variables and function symbol as follows:
- A word starting with a *lower* case letter is interpreted as a *variable*.
- A word starting with an *upper* case letter is assumed to be a *function* or *predicate symbol*.

In [2]:
import { LogicParser } from './FOL-Parser';
import { RecursiveSet } from './Recursive-Set';

In [3]:
type Variable = string;
type Formula = string | [string, ...Formula[]];
type Literal = string | [string, ...Formula[]];
type Clause = RecursiveSet<Literal>;
type CNF = RecursiveSet<Clause>;
type Substitution = Record<Variable, Formula>;
type LogicalExpression = Formula | Clause | CNF;

In [4]:
function formulaToString(f: LogicalExpression): string {
    if (typeof f === 'string') return f;
    if (f instanceof RecursiveSet) {
        return `{ ${[...f].map(e => formulaToString(e as LogicalExpression)).join(', ')} }`;
    }
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        if (op === '∀' || op === '∃') {
            const [variable, formula] = args as [string, Formula];
            return `${op}${variable}: ${formulaToString(formula)}`;
        }
        if (['∧', '∨', '→', '↔'].includes(op)) {
            const [left, right] = args as [Formula, Formula];
            return `(${formulaToString(left)} ${op} ${formulaToString(right)})`;
        }
        if (op === '¬') {
            return `¬${formulaToString(args[0] as Formula)}`;
        }
        if (args.length > 0) {
            return `${op}(${args.map(arg => formulaToString(arg as Formula)).join(', ')})`;
        }
        return op;
    }
    return String(f);
}

The function $\texttt{parse}(s)$ takes a string $s$ which is a formula from first order logic and turns this string into a 
*nested tuple*.

In [5]:
function parse(s: string): Formula {
    const p = new LogicParser(s);
    return p.parse();
}

For testing purposes, the following formula is used.  This formula specifies the notion of a *grandparent*.

In [6]:
const s = '∀g:∀c:(Grandparent(g, c) ↔ ∃p: (Parent(g, p) ∧ Parent(p, c)))';
const f1 = parse(s);
console.dir(f1, { depth: null }); 

[
  '∀',
  'g',
  [
    '∀',
    'c',
    [
      '↔',
      [ 'Grandparent', 'g', 'c' ],
      [
        '∃',
        'p',
        [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
      ]
    ]
  ]
]


The function $\texttt{apply}(t, σ)$ takes an object $t$ and a *variable substitution* $\sigma$ which is represented as a dictionary of the form $\{x_1: s_1, \cdots, x_n:s_n\}$ and replaces every occurrence of the variable $x_i$ in the object $t$ with the corresponding term $s_i$.  The object $t$ is either 
 - a term, 
 - a formula from first order logic (henceforth abbreviated as *FOL*), 
 - a clause (represented as a `set` of literals), or 
 - a set of clauses.

In [7]:
function apply(t: LogicalExpression, σ: Substitution): LogicalExpression {
    if (t instanceof RecursiveSet) {
        const newElements: any[] = [];
        for (const element of t) {
            newElements.push(apply(element as LogicalExpression, σ));
        }
        return new RecursiveSet(...newElements);
    }
    if (typeof t === 'string') {
        return Object.prototype.hasOwnProperty.call(σ, t) ? σ[t] : t;
    }
    if (Array.isArray(t)) {
        const [op, ...args] = t;
        const newArgs = args.map(arg => apply(arg, σ));
        return [op, ...newArgs] as Formula;
    }
    return t as Formula;
}

The assignment `f, *ts = t` shown above uses so called *extended iterable unpacking*.  The code below shows an example how this works.

In [8]:
const [f, ...ts] = ['parent', 'hugo', 'gustav'];

console.log([f, ts]);

[ 'parent', [ 'hugo', 'gustav' ] ]


In [9]:
console.dir(f1, { depth: null });

[
  '∀',
  'g',
  [
    '∀',
    'c',
    [
      '↔',
      [ 'Grandparent', 'g', 'c' ],
      [
        '∃',
        'p',
        [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
      ]
    ]
  ]
]


In [10]:
const result = apply(f1, { 'g': 'x', 'p': 'y', 'c': 'z' });
console.dir(result, { depth: null });

[
  '∀',
  'x',
  [
    '∀',
    'z',
    [
      '↔',
      [ 'Grandparent', 'x', 'z' ],
      [
        '∃',
        'y',
        [ '∧', [ 'Parent', 'x', 'y' ], [ 'Parent', 'y', 'z' ] ]
      ]
    ]
  ]
]


The function $\texttt{boundVariables}(f)$ computes the set of variables that are *bound* in the formula $f$. 

In [11]:
function boundVariables(f: Formula): RecursiveSet<string> {
    if (!Array.isArray(f)) {
        return new RecursiveSet();
    }
    const [op, ...args] = f;
    if (op === '∀' || op === '∃') {
        const [x, g] = args as [string, Formula];
        return new RecursiveSet(x).union(boundVariables(g));
    }        
    if (op === '⊤') return new RecursiveSet();
    if (op === '⊥') return new RecursiveSet();
    if (op === '¬') {
        const [g] = args as [Formula];
        return boundVariables(g);
    }
    if (['∧', '∨', '→', '↔'].includes(op)) {
        const [g, h] = args as [Formula, Formula];
        return boundVariables(g).union(boundVariables(h));
    }
    return new RecursiveSet(); 
}

In [12]:
console.dir(f1, { depth: null });

[
  '∀',
  'g',
  [
    '∀',
    'c',
    [
      '↔',
      [ 'Grandparent', 'g', 'c' ],
      [
        '∃',
        'p',
        [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
      ]
    ]
  ]
]


In [13]:
boundVariables(f1)

{c, g, p}


The function `allVariables` computes the set of all variables that occur in terms inside `f`. The object 
`f` is either a formula or a term.

In [14]:
function allVariables(f: Formula): RecursiveSet<string> {
    if (typeof f === 'string') {
        return new RecursiveSet(f);
    }
    const [op, ...args] = f;
    if (op === '∀' || op === '∃') {
        const [x, g] = args as [string, Formula];
        return new RecursiveSet(x).union(allVariables(g));
    }
    if (op === '⊤') return new RecursiveSet();
    if (op === '⊥') return new RecursiveSet();
    if (op === '¬') {
        const [g] = args as [Formula];
        return allVariables(g);
    }
    if (['∧', '∨', '→', '↔'].includes(op)) {
        const [g, h] = args as [Formula, Formula];
        return allVariables(g).union(allVariables(h));
    }
    let result = new RecursiveSet<string>();
    for (const t of args) {
        result = result.union(allVariables(t as Formula));
    }
    return result;
}

In [15]:
console.dir(f1, { depth: null });

[
  '∀',
  'g',
  [
    '∀',
    'c',
    [
      '↔',
      [ 'Grandparent', 'g', 'c' ],
      [
        '∃',
        'p',
        [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
      ]
    ]
  ]
]


In [16]:
allVariables(f1)

{c, g, p}


In [17]:
const g1: Formula = [
    '↔', 
    ['Grandparent', 'g', 'c'],
    ['∃', 'p', ['∧', ['Parent', 'g', 'p'], ['Parent', 'p', 'c']]]
];

In [18]:
allVariables(g1)

{c, g, p}


Below we use the `asciiLowercase` because it provides a definition of all lower case characters.

In [19]:
const asciiLowercase = 'abcdefghijklmnopqrstuvwxyz';
asciiLowercase;

abcdefghijklmnopqrstuvwxyz


In [20]:
const allLowercaseSet = new RecursiveSet(...asciiLowercase.split(''));

The function $\texttt{renameBoundVariables}(f)$ takes a first order formula $f$ and replaces all bound variables by **new** variables.  This only works if the set of characters `allLowercaseSet` has enough characters that do not already occur in $f$.  This approach would not be good enough for a production quality program,
but for the case of a demonstration it is sufficient.  The alternative would be to rename the variables as `x1`, `x2`, `x3`, $\cdots$, but that becomes unreadable very fast.

In [21]:
function renameBoundVariables(f: Formula): Formula {
    const boundVs = boundVariables(f);
    const allVs = allVariables(f);
    const availableVars = asciiLowercase.split('').filter(char => !allVs.has(char));
    const sigma: Substitution = {};
    let i = 0;
    for (const x of boundVs) {
        if (i < availableVars.length) {
            sigma[x] = availableVars[i];
            i++;
        } else {
            throw new Error("Not enough free variables available for renaming!");
        }
    }
    return apply(f, sigma) as Formula;
}

In [22]:
['a', 'b', 'c'].map((x, i) => [i, x])

[ [ 0, 'a' ], [ 1, 'b' ], [ 2, 'c' ] ]


In [23]:
console.dir(f1, { depth: null });

[
  '∀',
  'g',
  [
    '∀',
    'c',
    [
      '↔',
      [ 'Grandparent', 'g', 'c' ],
      [
        '∃',
        'p',
        [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
      ]
    ]
  ]
]


In [24]:
console.dir(renameBoundVariables(f1), { depth: null });

[
  '∀',
  'b',
  [
    '∀',
    'a',
    [
      '↔',
      [ 'Grandparent', 'b', 'a' ],
      [
        '∃',
        'd',
        [ '∧', [ 'Parent', 'b', 'd' ], [ 'Parent', 'd', 'a' ] ]
      ]
    ]
  ]
]


## Elimination Biconditionals

The function $\texttt{eliminateBiconditional}(f)$ takes a formula $f$ from first order logic and eliminates all occurrences of the operator '↔' from this formula.  This is done by using the following equivalence:
$$ (f \leftrightarrow g) \;\Leftrightarrow\; (f \rightarrow g) \wedge (g \rightarrow f) $$
In order to ensure that the resulting formula is <em style="color:blue">pure</em>, we have to rename the bound variables in the formula $g \rightarrow f$.

In [25]:
function eliminateBiconditional(f: Formula): Formula {
    if (typeof f === 'string') return f;
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        
        switch (op) {
            case '↔': {
                const [g, h] = args as [Formula, Formula];
                const ge = eliminateBiconditional(g);
                const he = eliminateBiconditional(h);
                return ['∧', ['→', ge, he], renameBoundVariables(['→', he, ge])] as Formula;
            }
            case '⊤':
            case '⊥':
                return f;
            case '¬': {
                const [g] = args as [Formula];
                return ['¬', eliminateBiconditional(g)];
            }
            case '∧':
            case '∨': 
            case '→': {
                const [g, h] = args as [Formula, Formula];
                return [op, eliminateBiconditional(g), eliminateBiconditional(h)];
            }
            case '∀':
            case '∃': {
                const [x, g] = args as [string, Formula];
                return [op, x, eliminateBiconditional(g)];
            }
        }
    }
    return f;
}

In [26]:
console.dir(f1, { depth: null });

[
  '∀',
  'g',
  [
    '∀',
    'c',
    [
      '↔',
      [ 'Grandparent', 'g', 'c' ],
      [
        '∃',
        'p',
        [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
      ]
    ]
  ]
]


In [27]:
const f2 = eliminateBiconditional(f1);
console.dir(f2, { depth: null });

[
  '∀',
  'g',
  [
    '∀',
    'c',
    [
      '∧',
      [
        '→',
        [ 'Grandparent', 'g', 'c' ],
        [
          '∃',
          'p',
          [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
        ]
      ],
      [
        '→',
        [
          '∃',
          'a',
          [ '∧', [ 'Parent', 'g', 'a' ], [ 'Parent', 'a', 'c' ] ]
        ],
        [ 'Grandparent', 'g', 'c' ]
      ]
    ]
  ]
]


## Eliminating Conditionals

The function $\texttt{eliminateConditional}(f)$ takes a formula $f$ from first order logic and eliminates all occurrences of the operator '→' from this formula.  This is done by using the following equivalence:
$$ (f \rightarrow g) \;\Leftrightarrow\; (\neg f \vee g) $$
The implementation of this function is similar to the implementation of the function `eliminateConditional` that we had used in propositional logic.

In [28]:
function eliminateConditional(f: Formula): Formula {
    if (typeof f === 'string') return f;
    if (Array.isArray(f)) {
        const [op, ...args] = f;

        switch (op) {
            case '→': {
                const [g, h] = args as [Formula, Formula];
                return ['∨', ['¬', eliminateConditional(g)], eliminateConditional(h)];
            }
            case '⊤':
            case '⊥':
                return f;
            case '¬': {
                const [g] = args as [Formula];
                return ['¬', eliminateConditional(g)];
            }
            case '∧':
            case '∨': {
                const [g, h] = args as [Formula, Formula];
                return [op, eliminateConditional(g), eliminateConditional(h)];
            }
            case '∀':
            case '∃': {
                const [x, g] = args as [string, Formula];
                return [op, x, eliminateConditional(g)];
            }
            default:
                return f;
        }
    }
    return f;
}

In [29]:
console.dir(f2, { depth: null });

[
  '∀',
  'g',
  [
    '∀',
    'c',
    [
      '∧',
      [
        '→',
        [ 'Grandparent', 'g', 'c' ],
        [
          '∃',
          'p',
          [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
        ]
      ],
      [
        '→',
        [
          '∃',
          'a',
          [ '∧', [ 'Parent', 'g', 'a' ], [ 'Parent', 'a', 'c' ] ]
        ],
        [ 'Grandparent', 'g', 'c' ]
      ]
    ]
  ]
]


In [30]:
const f3 = eliminateConditional(f2);
console.dir(f3, { depth: null });

[
  '∀',
  'g',
  [
    '∀',
    'c',
    [
      '∧',
      [
        '∨',
        [ '¬', [ 'Grandparent', 'g', 'c' ] ],
        [
          '∃',
          'p',
          [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
        ]
      ],
      [
        '∨',
        [
          '¬',
          [
            '∃',
            'a',
            [ '∧', [ 'Parent', 'g', 'a' ], [ 'Parent', 'a', 'c' ] ]
          ]
        ],
        [ 'Grandparent', 'g', 'c' ]
      ]
    ]
  ]
]


## Negation Normal Form

The function $\texttt{nnf}(f)$ computes the <em style="color:blue;">negation normal form</em> of $f$, while $\texttt{neg}(f)$ computes the *negation normal form* of $\neg f$.  The expression $\texttt{nnf}(f)$ is defined recursively as follows:
<ol>
    <li> $\texttt{nnf}(\neg \texttt{F}) = \texttt{neg}(\texttt{F})$, </li>
    <li> $\texttt{nnf}(\texttt{F}_1 \wedge \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \wedge \texttt{nnf}(\texttt{F}_2)$,</li>
    <li> $\texttt{nnf}(\texttt{F}_1 \vee \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \vee \texttt{nnf}(\texttt{F}_2)$.</li>
    <li> $\texttt{nnf}(\forall x: F) = \forall x: \texttt{nnf}(\texttt{F})$.</li>
    <li> $\texttt{nnf}(\exists x: F) = \exists x: \texttt{nnf}(\texttt{F})$.</li>
</ol>

The forward declaration for the function `neg` is needed to typecheck the function `nnf`.

In [31]:
let neg = (f: Formula): Formula => {
    return f;
};

In [32]:
function nnf(f: Formula): Formula {
    if (typeof f === 'string') return f;
    if (Array.isArray(f)) {
        const [op, ...args] = f;

        switch (op) {
            case '⊤':
            case '⊥':
                return f;
            case '¬': {
                const [g] = args as [Formula];
                return neg(g);
            }
            case '∧':
            case '∨': {
                const [g, h] = args as [Formula, Formula];
                return [op, nnf(g), nnf(h)];
            }
            case '∀':
            case '∃': {
                const [x, g] = args as [string, Formula];
                return [op, x, nnf(g)];
            }
        }
    }
    return f;
}

The auxiliary function $\texttt{neg}$ is also defined recursively:
<ol>
    <li> $\texttt{neg}(p) = \texttt{nnf}(\neg p) = \neg p$ for all propositional variables $p$,</li>
    <li> $\texttt{neg}(\neg F) = \texttt{nnf}(\neg \neg F) = \texttt{nnf}(F)$,</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \wedge F_2 \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \wedge F_2)\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \vee \neg F_2\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \vee \texttt{nnf}\bigl(\neg F_2\bigr) \\[0.1cm]
       = & \texttt{neg}(F_1) \vee \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \wedge F_2 \bigr) = \texttt{neg}(F_1) \vee \texttt{neg}(F_2)$.</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \vee F_2 \bigr)        \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \vee F_2) \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \wedge \neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \wedge \texttt{nnf}\bigl(\neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{neg}(F_1) \wedge \texttt{neg}(F_2). 
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \vee F_2 \bigr) = \texttt{neg}(F_1) \wedge \texttt{neg}(F_2)$.</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(\forall x: F \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg \forall x: F\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\exists x: \neg F\bigr) \\[0.1cm]
       = & \exists x: \texttt{nnf}(\neg F)           \\[0.1cm]
       = & \exists x: \texttt{neg}(F).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(\forall x: F \bigr) = \exists x: \texttt{neg}(F)$.</li>
      <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(\exists x: F \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg \exists x: F\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\forall x: \neg F\bigr) \\[0.1cm]
       = & \forall x: \texttt{nnf}(\neg F)           \\[0.1cm]
       = & \forall x: \texttt{neg}(F).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(\exists x: F \bigr) = \forall x: \texttt{neg}(F)$.</li>
</ol>

In [33]:
neg = function(f: Formula): Formula {
    if (typeof f === 'string') return ['¬', f];
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        
        switch (op) {
            case '⊤':
                return ['⊥'];
            case '⊥':
                return ['⊤'];

            case '¬': {
                const [g] = args as [Formula];
                return nnf(g);
            }
            case '∧': {
                const [g, h] = args as [Formula, Formula];
                return ['∨', neg(g), neg(h)];
            }
            case '∨': {
                const [g, h] = args as [Formula, Formula];
                return ['∧', neg(g), neg(h)];
            }
            case '∀': {
                const [x, g] = args as [string, Formula];
                return ['∃', x, neg(g)];
            }
            case '∃': {
                const [x, g] = args as [string, Formula];
                return ['∀', x, neg(g)];
            }
            default:
                return ['¬', f];
        }
    }
    return ['¬', f];
}

[Function (anonymous)]


In [34]:
console.dir(f3, { depth: null });

[
  '∀',
  'g',
  [
    '∀',
    'c',
    [
      '∧',
      [
        '∨',
        [ '¬', [ 'Grandparent', 'g', 'c' ] ],
        [
          '∃',
          'p',
          [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
        ]
      ],
      [
        '∨',
        [
          '¬',
          [
            '∃',
            'a',
            [ '∧', [ 'Parent', 'g', 'a' ], [ 'Parent', 'a', 'c' ] ]
          ]
        ],
        [ 'Grandparent', 'g', 'c' ]
      ]
    ]
  ]
]


In [35]:
const f4 = nnf(f3)
console.dir(f4, { depth: null });

[
  '∀',
  'g',
  [
    '∀',
    'c',
    [
      '∧',
      [
        '∨',
        [ '¬', [ 'Grandparent', 'g', 'c' ] ],
        [
          '∃',
          'p',
          [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
        ]
      ],
      [
        '∨',
        [
          '∀',
          'a',
          [
            '∨',
            [ '¬', [ 'Parent', 'g', 'a' ] ],
            [ '¬', [ 'Parent', 'a', 'c' ] ]
          ]
        ],
        [ 'Grandparent', 'g', 'c' ]
      ]
    ]
  ]
]


## Prenex Normal Form

In the following we assume that all quantifiers that occur in a formula bind **different** variables, i.e. we assume that the formulas are *pure*.  If this assumption is not satisfied, then the functions given below will produce <u>garbage</u>.

A *quantifier tuple* is a tuple of the following form:
$$ (Q_1, x_1, \cdots, Q_n, x_n) $$
Here, the $Q_i$ denote quantifiers, i.e. we have $Q_i \in \{\forall, \exists\}$, while the $x_i$ are variables.  The function $\texttt{mergeQuantifiers}(T_1, T_2)$ takes two quantifier tuples $T_1$ and $T_2$ as arguments and merges them into a new quantifier tuple such that the relative order of the quantifiers remains the same, i.e. if both $Q_1, x_1$ and $Q_2, x_2$ occur in $T_1$ and $Q_1, x_1$ occurs before $Q_2, x_2$, then $Q_1, x_1$ will occur before $Q_2, x_2$ in the result.

In [36]:
function mergeQuantifiers(Q1: string[], Q2: string[]): string[] {
    if (Q1.length === 0) return Q2;
    if (Q2.length === 0) return Q1;

    if (Q1[0] === '∃') {
        return [...Q1.slice(0, 2), ...mergeQuantifiers(Q1.slice(2), Q2)];
    }

    if (Q2[0] === '∃') {
        return [...Q2.slice(0, 2), ...mergeQuantifiers(Q1, Q2.slice(2))];
    }

    return [...Q1.slice(0, 2), ...mergeQuantifiers(Q1.slice(2), Q2)];
}

In [37]:
const resultQ = mergeQuantifiers(
    ['∀', 'x', '∃', 'y'], 
    ['∃', 'u', '∀', 'v']
);

console.log(resultQ);

[
  '∃', 'u', '∀',
  'x', '∃', 'y',
  '∀', 'v'
]


Given a formula $f$, the function $\texttt{extractQuantifiers}(f)$ returns a pairs $(T, m)$, where $T$ is a quantifier tuple and $m$ is the <em style="color:blue;">matrix</em> of the formula $f$, where the matrix of a formula is defined as the part that remains when all quantifiers have been extracted.

In [38]:
function extractQuantifiers(f: Formula): [string[], Formula] {
    if (typeof f === 'string') return [[], f];
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        
        switch (op) {
            case '⊤':
            case '⊥':
                return [[], f];
            case '¬': 
                return [[], f];

            case '∧':
            case '∨': {
                const [g, h] = args as [Formula, Formula];
                const [qg, gm] = extractQuantifiers(g);
                const [qh, hm] = extractQuantifiers(h);
                const matrix: Formula = [op, gm, hm];
                return [mergeQuantifiers(qg, qh), matrix];
            }
            case '∀':
            case '∃': {
                const [x, g] = args as [string, Formula];
                const [qg, gm] = extractQuantifiers(g);
                const newQuantifiers = [op, x, ...qg];
                return [newQuantifiers, gm];
            }
            default:
                return [[], f];
        }
    }
    return [[], f];
}

In [39]:
console.dir(f4, { depth: null });

[
  '∀',
  'g',
  [
    '∀',
    'c',
    [
      '∧',
      [
        '∨',
        [ '¬', [ 'Grandparent', 'g', 'c' ] ],
        [
          '∃',
          'p',
          [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
        ]
      ],
      [
        '∨',
        [
          '∀',
          'a',
          [
            '∨',
            [ '¬', [ 'Parent', 'g', 'a' ] ],
            [ '¬', [ 'Parent', 'a', 'c' ] ]
          ]
        ],
        [ 'Grandparent', 'g', 'c' ]
      ]
    ]
  ]
]


In [40]:
const [Qs, f5] = extractQuantifiers(f4);

console.log('Quantifiers (Qs):', Qs);
console.dir(f5, { depth: null });

Quantifiers (Qs): [
  '∀', 'g', '∀',
  'c', '∃', 'p',
  '∀', 'a'
]
[
  '∧',
  [
    '∨',
    [ '¬', [ 'Grandparent', 'g', 'c' ] ],
    [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
  ],
  [
    '∨',
    [
      '∨',
      [ '¬', [ 'Parent', 'g', 'a' ] ],
      [ '¬', [ 'Parent', 'a', 'c' ] ]
    ],
    [ 'Grandparent', 'g', 'c' ]
  ]
]


Given a qantifier tuple $\texttt{Qs}$ and a matrix $m$, the call $\texttt{attachQuantifiers}(Qs, m)$ combines the quantifiers $\texttt{Qs}$ and the matrix $m$ into a quantified formula.

In [41]:
function attachQuantifiers(Qs: string[], m: Formula): Formula {
    if (Qs.length === 0) return m;
    const Q = Qs[0];
    const x = Qs[1];
    const Qr = Qs.slice(2);
    return [Q, x, attachQuantifiers(Qr, m)];
}

In [42]:
Qs

[
  '∀', 'g', '∀',
  'c', '∃', 'p',
  '∀', 'a'
]


In [43]:
console.dir(f5, { depth: null });

[
  '∧',
  [
    '∨',
    [ '¬', [ 'Grandparent', 'g', 'c' ] ],
    [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
  ],
  [
    '∨',
    [
      '∨',
      [ '¬', [ 'Parent', 'g', 'a' ] ],
      [ '¬', [ 'Parent', 'a', 'c' ] ]
    ],
    [ 'Grandparent', 'g', 'c' ]
  ]
]


In [44]:
const f6 = attachQuantifiers(Qs, f5)
console.dir(f6, { depth: null });

[
  '∀',
  'g',
  [
    '∀',
    'c',
    [
      '∃',
      'p',
      [
        '∀',
        'a',
        [
          '∧',
          [
            '∨',
            [ '¬', [ 'Grandparent', 'g', 'c' ] ],
            [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
          ],
          [
            '∨',
            [
              '∨',
              [ '¬', [ 'Parent', 'g', 'a' ] ],
              [ '¬', [ 'Parent', 'a', 'c' ] ]
            ],
            [ 'Grandparent', 'g', 'c' ]
          ]
        ]
      ]
    ]
  ]
]


## Skolemization (Eliminating Existential Quantifiers)

The variable $\texttt{skolemCounter}$ is a global variable that is needed to create unique Skolem constants.  

In [45]:
let skolemCounter = 0

In [46]:
function skolemConstant(): string {
    skolemCounter += 1;
    return 'sk' + skolemCounter;
}

The function $\texttt{skolemize}(f, \texttt{Vs})$ takes a formula $f$ and a tuple of variables $\texttt{Vs}$ and 
<em style="color:blue">skolemizes</em> the formula $f$, i.e. it replaces all existentially quantified variables by appropriate <em style="color:blue">Skolem functions</em>.  The tuple $\texttt{Vs}$ is a tuple of variables that are 
assumed to be universally quantified.  The formula $f$ is assumed to be in <em style="color:blue">prenex normal form</em>.

For skolemization to work correctly, we have to assume that 
<font size="4" style="color:darkgreen; size:125%">$f$ does not contain free variables</font>!

In [47]:
function skolemize(f: Formula, Vs: string[]): Formula {
    if (typeof f === 'string') return f;
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        
        switch (op) {
            case '∃': {
                const [x, g] = args as [string, Formula];
                const skolemFunc = skolemConstant();
                const t: Formula = [skolemFunc, ...Vs]; 
                const sigma: Substitution = { [x]: t };
                const appliedG = apply(g, sigma) as Formula;
                return skolemize(appliedG, Vs);
            }
            case '∀': {
                const [x, g] = args as [string, Formula];
                return skolemize(g, [...Vs, x]);
            }
            default:
                return f;
        }
    }
    return f;
}

In [48]:
console.dir(f6, { depth: null });

[
  '∀',
  'g',
  [
    '∀',
    'c',
    [
      '∃',
      'p',
      [
        '∀',
        'a',
        [
          '∧',
          [
            '∨',
            [ '¬', [ 'Grandparent', 'g', 'c' ] ],
            [ '∧', [ 'Parent', 'g', 'p' ], [ 'Parent', 'p', 'c' ] ]
          ],
          [
            '∨',
            [
              '∨',
              [ '¬', [ 'Parent', 'g', 'a' ] ],
              [ '¬', [ 'Parent', 'a', 'c' ] ]
            ],
            [ 'Grandparent', 'g', 'c' ]
          ]
        ]
      ]
    ]
  ]
]


In [49]:
const f7 = skolemize(f6, []);
console.dir(f7, { depth: null });

[
  '∧',
  [
    '∨',
    [ '¬', [ 'Grandparent', 'g', 'c' ] ],
    [
      '∧',
      [ 'Parent', 'g', [ 'sk1', 'g', 'c' ] ],
      [ 'Parent', [ 'sk1', 'g', 'c' ], 'c' ]
    ]
  ],
  [
    '∨',
    [
      '∨',
      [ '¬', [ 'Parent', 'g', 'a' ] ],
      [ '¬', [ 'Parent', 'a', 'c' ] ]
    ],
    [ 'Grandparent', 'g', 'c' ]
  ]
]


## Conversion to Clauses

The function $\texttt{cnf}(f)$ takes a <em style="color:blue">skolemized</em> formula $f$ from first order logic that is in <em style="color:blue">negation normal form</em> and returns the <em style="color:blue">conjunctive normal form</em> of $f$ in <em style="color:blue">set notation</em>.  This works the same way as in propositional logic.

In [50]:
function cnf(f: Formula): CNF {
    if (typeof f === 'string') {
        return new RecursiveSet(new RecursiveSet(f));
    }
    if (Array.isArray(f)) {
        const [op, ...args] = f;
        
        switch (op) {
            case '⊤':
                return new RecursiveSet();
            case '⊥':
                return new RecursiveSet(new RecursiveSet());

            case '¬': {
                return new RecursiveSet(new RecursiveSet(f as Literal)); 
            }
            case '∧': {
                const [g, h] = args as [Formula, Formula];
                return cnf(g).union(cnf(h));
            }
            case '∨': {
                const [g, h] = args as [Formula, Formula];
                const cnfG = cnf(g);
                const cnfH = cnf(h);
                const result = new RecursiveSet<Clause>();
                for (const k1 of cnfG) {
                    for (const k2 of cnfH) {
                        result.add(k1.union(k2));
                    }
                }
                return result;
            }
            default:
                return new RecursiveSet(new RecursiveSet(f as unknown as Literal));
        }
    }
    return new RecursiveSet(new RecursiveSet(f as unknown as Literal));
}

In [51]:
console.dir(f7, { depth: null });

[
  '∧',
  [
    '∨',
    [ '¬', [ 'Grandparent', 'g', 'c' ] ],
    [
      '∧',
      [ 'Parent', 'g', [ 'sk1', 'g', 'c' ] ],
      [ 'Parent', [ 'sk1', 'g', 'c' ], 'c' ]
    ]
  ],
  [
    '∨',
    [
      '∨',
      [ '¬', [ 'Parent', 'g', 'a' ] ],
      [ '¬', [ 'Parent', 'a', 'c' ] ]
    ],
    [ 'Grandparent', 'g', 'c' ]
  ]
]


In [52]:
const f8 = cnf(f7);
formulaToString(f8);

{ { Parent(g, sk1(g, c)), ¬Grandparent(g, c) }, { Parent(sk1(g, c), c), ¬Grandparent(g, c) }, { Grandparent(g, c), ¬Parent(a, c), ¬Parent(g, a) } }


## Putting Everything Together

The function $f$ takes a <em style="color:blue">pure</em> formula $f$ from first order logic and transforms $f$ into a set of first order clauses.  Furthermore, $f$ **must not** contain free variables.

In [53]:
function normalize(f: Formula): CNF {
    const f1 = eliminateBiconditional(f);
    const f2 = eliminateConditional(f1);
    const f3 = nnf(f2);
    const [Qs, f4] = extractQuantifiers(f3);
    const f5 = attachQuantifiers(Qs, f4);
    const f6 = skolemize(f5, []);
    const f7 = cnf(f6);
    return f7;
}

In [54]:
formulaToString(normalize(f1));

{ { Parent(g, sk2(g, c)), ¬Grandparent(g, c) }, { Parent(sk2(g, c), c), ¬Grandparent(g, c) }, { Grandparent(g, c), ¬Parent(a, c), ¬Parent(g, a) } }


In [55]:
function prettify(M: CNF): string {
    if (M.size === 0) {
        return "{}";
    }
    let result = "{\n";
    for (const A of M) {
        const clause = A as RecursiveSet<Literal>;
        if (clause.size === 0) {
            result += "  {},\n";
        } else {
            const literals = Array.from(clause)
                .map(l => formulaToString(l as Formula))
                .sort();           
            result += `  { ${literals.join(', ')} },\n`;
        }
    }
    result = result.substring(0, result.length - 2); 
    result += "\n}";   
    return result;
}

In [56]:
function test(s: string): void {
    const f = new LogicParser(s).parse();
    console.log(`The knf of ${s} is:`);
    console.log(prettify(normalize(f)));
}

In [57]:
test(s);

The knf of ∀g:∀c:(Grandparent(g, c) ↔ ∃p: (Parent(g, p) ∧ Parent(p, c))) is:
{
  { Parent(g, sk3(g, c)), ¬Grandparent(g, c) },
  { Parent(sk3(g, c), c), ¬Grandparent(g, c) },
  { Grandparent(g, c), ¬Parent(a, c), ¬Parent(g, a) }
}


In [58]:
test('¬(∃y:∀x:P(x,y)→∀u:∃v:P(u,v))');

The knf of ¬(∃y:∀x:P(x,y)→∀u:∃v:P(u,v)) is:
{
  { P(x, sk4) },
  { ¬P(sk5, v) }
}
